# Notebook 48R1 - Hybrid Frontier Test
Creates Hybrid25/50/75, Median, Minimum and Maximum ensembles from R1_biharmonic and R3_learned_unet outputs and prepares scoring using Notebook 46 helpers.


In [5]:
from google.colab import drive
drive.mount('/content/drive')
import json, cv2, numpy as np, pandas as pd
from pathlib import Path
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
RUN=max([p for p in (ROOT/'observed_print_classifier_v8_6_notebook46').iterdir() if p.is_dir()], key=lambda p:p.stat().st_mtime)
TABLES=RUN/'tables'
OUT=RUN/'notebook48R1_hybrid_frontier'
HYBRIDS=OUT/'hybrids'
FIGS=OUT/'figures'
for p in [OUT,HYBRIDS,FIGS]: p.mkdir(parents=True,exist_ok=True)
inventory=pd.read_csv(TABLES/'whole_page_candidate_inventory.csv')
print(RUN)


/content/drive/MyDrive/MusicAnnotationProject/observed_print_classifier_v8_6_notebook46/20261021_131744_v8_6_notebook46


In [3]:
def ink(im):
    return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):
    return cv2.Canny(im,70,170)>0
def f1_from_features(ref,pred,reg,tol=1):
    ref=ref&reg; pred=pred&reg
    if not ref.any(): return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8)
    p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1)
    q=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum()
    return float(p),float(q),float(2*p*q/max(p+q,1e-12))


In [4]:
pairs=inventory[inventory.restorer.isin(['R1_biharmonic','R3_learned_unet'])]
pairs=(pairs.pivot_table(index=['page','source_group','arm'],columns='restorer',values='candidate',aggfunc='first').reset_index())
rows=[]
for _,r in pairs.iterrows():
    b=cv2.imread(str(r['R1_biharmonic']),0)
    u=cv2.imread(str(r['R3_learned_unet']),0)
    if b is None or u is None: continue
    products={
      'Hybrid25':0.75*b+0.25*u,
      'Hybrid50':0.5*b+0.5*u,
      'Hybrid75':0.25*b+0.75*u,
      'Median':np.median(np.stack([b,u]),axis=0),
      'Minimum':np.minimum(b,u),
      'Maximum':np.maximum(b,u)}
    for name,img in products.items():
      img=np.clip(img,0,255).astype(np.uint8)
      fp=HYBRIDS/f'{r.page}_{r.arm}_{name}.png'
      cv2.imwrite(str(fp),img)
      rows.append({'page':r.page,'source_group':r.source_group,'arm':r.arm,'restorer':name,'candidate':str(fp)})
hybrid_inventory=pd.DataFrame(rows)
hybrid_inventory.to_csv(OUT/'hybrid_inventory.csv',index=False)
print('Generated',len(hybrid_inventory),'hybrids')


Generated 1290 hybrids


Copy the page_clean, page_obs and page_auth setup cells from Notebook 46 immediately above the scoring stage, then score hybrid_inventory using the same whole-page scoring block, replacing whole_candidates with hybrid_inventory.


In [7]:
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):
            h.update(b)
    return h.hexdigest()

def atomic_json(path, value):
    path=Path(path); tmp=Path(str(path)+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str))
    tmp.replace(path)

def atomic_csv(frame, path):
    path=Path(path); tmp=Path(str(path)+'.tmp')
    frame.to_csv(tmp,index=False); tmp.replace(path)

def latest(root,req):
    hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in req)] if Path(root).exists() else []
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

REQ45=['mask_lock.json','notebook45_verdict.json','source_record.json','tables/region_manifest.parquet','tables/mask_readiness_metrics.csv']
N45=latest(N45_ROOT,REQ45)
if N45 is None:
    raise RuntimeError('No complete Notebook 45 run found.')

lock=json.loads((N45/'mask_lock.json').read_text())
verdict45=json.loads((N45/'notebook45_verdict.json').read_text())
source=json.loads((N45/'source_record.json').read_text())
regions=pd.read_parquet(N45/'tables/region_manifest.parquet')

record={
    'notebook45':str(N45),
    'mask_lock_sha256':sha(N45/'mask_lock.json'),
    'mask_lock_id':lock['lock_id'],
    'source_record_sha256':sha(N45/'source_record.json'),
    'notebook45_ready':verdict45.get('ready_for_crossover'),
}

source_record_path=OUT/'source_record.json'
if source_record_path.exists():
    previous=json.loads(source_record_path.read_text())
    for key in ['mask_lock_sha256','mask_lock_id','source_record_sha256']:
        if previous.get(key)!=record.get(key):
            raise RuntimeError(f'Resume blocked: {key} differs from the crashed run.')
else:
    atomic_json(source_record_path,record)


print(json.dumps(record,indent=2))
print('Regions:',len(regions),'Pages:',regions.page.nunique(),'Source groups:',regions.source_group.nunique())

NameError: name 'N45_ROOT' is not defined

In [ ]:



source_v82 = Path(source['source_v82'])

clean_map={
    p.stem: p for p in (ROOT/'source'/'clean').iterdir()
    if p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
}

page_obs={
    p:cv2.imread(
        str(source_v82/'whole_pages'/p/'observed.png'),0)
    for p in regions.page.unique()
}

page_clean={
    p:cv2.imread(
        str(clean_map[p]),0)
    for p in regions.page.unique()
}

In [ ]:
# Restore post-output runtime state removed with the original whole-page assembly cells.
# This reconstructs page_auth exactly as 46R5 did, from the V7.5 authoritative mask cache.
source45=json.loads((N45/'source_record.json').read_text())
v75=Path(source45['v75'])
v75_status=pd.read_csv(v75/'tables/cache_status.csv')
v75_cache={str(r.page):Path(r.cache) for _,r in v75_status.iterrows()}
page_auth={}
for page in sorted(regions.page.unique()):
    if page not in v75_cache:
        raise RuntimeError(f'{page}: V7.5 cache missing while restoring page_auth')
    with np.load(v75_cache[page]) as z:
        if 'mask' not in z.files:
            raise RuntimeError(f'{page}: V7.5 cache missing authoritative mask')
        auth=z['mask']>0
    obs=page_obs[page]
    if obs is None or auth.shape!=obs.shape:
        raise RuntimeError(f'{page}: observed/authoritative-mask shape mismatch')
    page_auth[page]=auth

candidate_inventory_path=TABLES/'whole_page_candidate_inventory.csv'
if not candidate_inventory_path.exists():
    raise RuntimeError(f'Missing upstream whole-page candidate inventory: {candidate_inventory_path}')
print('Restored page_auth for',len(page_auth),'pages; whole-page candidate inventory present.')


In [ ]:
# Whole-page candidate scoring. Clean truth is used only here, after candidate assembly.
hybrid_inventory=pd.read_csv(TABLES/'whole_page_candidate_inventory.csv')
whole_metric_rows=[]
for _,row in hybrid_inventory.iterrows():
    page=row.page; clean=page_clean[page]; obs=page_obs[page]; auth=page_auth[page]
    pred=cv2.imread(str(row.candidate),cv2.IMREAD_GRAYSCALE)
    if pred is None or pred.shape!=clean.shape: raise RuntimeError(f'Invalid whole-page candidate: {row.candidate}')
    diff=np.abs(clean.astype(np.float32)-pred.astype(np.float32))
    _,sm=ssim(clean,pred,data_range=255,full=True)
    ip,ir,if1=f1_from_features(ink(clean),ink(pred),auth); ep,er,ef1=f1_from_features(edges(clean),edges(pred),auth)
    whole_metric_rows.append({
        'page':page,'source_group':row.source_group,'arm':row.arm,'restorer':row.restorer,
        'authoritative_pixels':int(auth.sum()),'mae_authoritative':float(diff[auth].mean()) if auth.any() else np.nan,
        'ssim_authoritative':float(sm[auth].mean()) if auth.any() else np.nan,
        'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,
        'false_inserted_ink':int((ink(pred)&~ink(clean)&auth).sum()),
        'missing_ink':int((ink(clean)&~ink(pred)&auth).sum()),
        'outside_mask_drift':int(np.sum(pred[~auth]!=obs[~auth])),
        'changed_vs_observed':int(np.sum(pred!=obs)),
    })
whole_page_metrics=pd.DataFrame(whole_metric_rows)
if len(whole_page_metrics)!=len(hybrid_inventory): raise RuntimeError('Whole-page metric inventory incomplete.')
atomic_csv(whole_page_metrics,TABLES/'whole_page_candidate_metrics.csv')
atomic_csv(whole_page_metrics.groupby(['arm','restorer'],as_index=False).agg(
    pages=('page','nunique'),mae_authoritative=('mae_authoritative','mean'),ssim_authoritative=('ssim_authoritative','mean'),
    ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),
    missing_ink=('missing_ink','sum'),outside_mask_drift=('outside_mask_drift','sum')
),TABLES/'whole_page_candidate_summary.csv')
print('WHOLE-PAGE SCORING COMPLETE:',len(whole_page_metrics))